# CO₂ Depressurization and Blowdown

This notebook simulates the depressurization (blowdown) of CO₂-rich systems,
critical for CCS pipeline and vessel safety design.

## Why Is CO₂ Depressurization Different?

CO₂ has unique properties that make depressurization challenging:

- **Triple point** at 5.18 bara / -56.6°C — solid CO₂ (dry ice) can form
- **Joule-Thomson cooling** is severe — temperatures can drop below -80°C
- **Phase transitions** — dense phase → liquid → gas → solid possible during blowdown
- **Brittle fracture risk** — extreme cold can cause pipeline failure

## Topics Covered
- Isenthalpic (Joule-Thomson) expansion of CO₂
- Isentropic expansion cooling curves
- Staged depressurization simulation
- Phase behavior during blowdown

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Separator = jneqsim.process.equipment.separator.Separator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

print('NeqSim loaded successfully')

## 1. Joule-Thomson Cooling of CO₂

When CO₂ expands through a valve (isenthalpic), it cools dramatically.
Calculate the outlet temperature vs outlet pressure for different inlet conditions.

In [ ]:
# JT expansion from different initial conditions
inlet_conditions = [
    (40.0, 150.0, 'T₀=40°C, P₀=150 bar'),
    (25.0, 100.0, 'T₀=25°C, P₀=100 bar'),
    (10.0, 80.0,  'T₀=10°C, P₀=80 bar'),
    (5.0,  60.0,  'T₀=5°C, P₀=60 bar'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ['blue', 'green', 'orange', 'red']

for (T_in, P_in, label), color in zip(inlet_conditions, colors):
    outlet_pressures = np.arange(6, P_in, 2)
    outlet_temps = []
    n_phases = []

    for P_out in outlet_pressures:
        try:
            co2 = SystemSrkEos(273.15 + T_in, P_in)
            co2.addComponent('CO2', 0.98)
            co2.addComponent('nitrogen', 0.02)
            co2.setMixingRule('classic')

            feed = Stream('Feed', co2)
            feed.setFlowRate(100000.0, 'kg/hr')
            feed.setTemperature(T_in, 'C')
            feed.setPressure(P_in, 'bara')

            valve = ThrottlingValve('JT Valve', feed)
            valve.setOutletPressure(float(P_out))

            proc = ProcessSystem()
            proc.add(feed)
            proc.add(valve)
            proc.run()

            T_out = float(valve.getOutletStream().getTemperature()) - 273.15
            nph = int(valve.getOutletStream().getFluid().getNumberOfPhases())
            outlet_temps.append(T_out)
            n_phases.append(nph)
        except Exception:
            outlet_temps.append(float('nan'))
            n_phases.append(0)

    ax1.plot(outlet_pressures, outlet_temps, '-', color=color, linewidth=2, label=label)

# Mark critical regions
ax1.axhline(y=-56.6, color='purple', linestyle=':', alpha=0.7, label='CO₂ triple point (-56.6°C)')
ax1.axhline(y=-46.0, color='gray', linestyle=':', alpha=0.7, label='MDMT limit (-46°C typical)')
ax1.set_xlabel('Outlet Pressure (bara)', fontsize=12)
ax1.set_ylabel('Outlet Temperature (°C)', fontsize=12)
ax1.set_title('Joule-Thomson Expansion of CO₂')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Isentropic expansion at P0=150 bara, T0=40°C
P_out_range = np.arange(6, 150, 2)
T_isentropic = []
T_jt = []

for P_out in P_out_range:
    try:
        # Isentropic
        co2_s = SystemSrkEos(273.15 + 40.0, 150.0)
        co2_s.addComponent('CO2', 0.98)
        co2_s.addComponent('nitrogen', 0.02)
        co2_s.setMixingRule('classic')

        ops = ThermodynamicOperations(co2_s)
        ops.TPflash()
        co2_s.initProperties()
        S_init = float(co2_s.getEntropy())

        # Find T at P_out with same entropy (isentropic)
        co2_out = SystemSrkEos(273.15 + 40.0, float(P_out))
        co2_out.addComponent('CO2', 0.98)
        co2_out.addComponent('nitrogen', 0.02)
        co2_out.setMixingRule('classic')

        ops2 = ThermodynamicOperations(co2_out)
        ops2.PSflash(S_init)
        co2_out.initProperties()
        T_isentropic.append(float(co2_out.getTemperature()) - 273.15)
    except Exception:
        T_isentropic.append(float('nan'))

    try:
        # JT (isenthalpic)
        co2_h = SystemSrkEos(273.15 + 40.0, 150.0)
        co2_h.addComponent('CO2', 0.98)
        co2_h.addComponent('nitrogen', 0.02)
        co2_h.setMixingRule('classic')

        f2 = Stream('Feed', co2_h)
        f2.setFlowRate(100000.0, 'kg/hr')
        f2.setTemperature(40.0, 'C')
        f2.setPressure(150.0, 'bara')
        v2 = ThrottlingValve('V', f2)
        v2.setOutletPressure(float(P_out))
        p2 = ProcessSystem()
        p2.add(f2)
        p2.add(v2)
        p2.run()
        T_jt.append(float(v2.getOutletStream().getTemperature()) - 273.15)
    except Exception:
        T_jt.append(float('nan'))

ax2.plot(P_out_range, T_isentropic, 'b-', linewidth=2, label='Isentropic (worst case)')
ax2.plot(P_out_range, T_jt, 'r-', linewidth=2, label='Isenthalpic (JT valve)')
ax2.axhline(y=-56.6, color='purple', linestyle=':', alpha=0.7, label='Triple point')
ax2.set_xlabel('Outlet Pressure (bara)', fontsize=12)
ax2.set_ylabel('Temperature (°C)', fontsize=12)
ax2.set_title('Isentropic vs Isenthalpic Expansion (P₀=150 bar, T₀=40°C)')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('co2_depressurization.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Staged Depressurization

To avoid extremely low temperatures, CO₂ systems can be depressurized in stages.
This shows the temperature at each stage.

In [ ]:
# Staged depressurization simulation
P_initial = 150.0  # bara
T_initial = 40.0   # °C
n_stages = 8

# Calculate pressure levels (geometric spacing)
P_final = 10.0
P_levels = np.geomspace(P_initial, P_final, n_stages + 1)

stage_data = []
T_current = T_initial

for i in range(n_stages):
    P_in = P_levels[i]
    P_out = P_levels[i + 1]

    try:
        co2 = SystemSrkEos(273.15 + T_current, float(P_in))
        co2.addComponent('CO2', 0.98)
        co2.addComponent('nitrogen', 0.02)
        co2.setMixingRule('classic')

        feed = Stream('Feed', co2)
        feed.setFlowRate(100000.0, 'kg/hr')
        feed.setTemperature(T_current, 'C')
        feed.setPressure(float(P_in), 'bara')

        valve = ThrottlingValve('Stage Valve', feed)
        valve.setOutletPressure(float(P_out))

        proc = ProcessSystem()
        proc.add(feed)
        proc.add(valve)
        proc.run()

        T_out = float(valve.getOutletStream().getTemperature()) - 273.15
        nph = int(valve.getOutletStream().getFluid().getNumberOfPhases())

        stage_data.append({
            'Stage': i + 1,
            'P_in (bara)': round(float(P_in), 1),
            'P_out (bara)': round(float(P_out), 1),
            'T_in (°C)': round(T_current, 1),
            'T_out (°C)': round(T_out, 1),
            'dT (°C)': round(T_out - T_current, 1),
            'Phases': nph
        })
        T_current = T_out
    except Exception as e:
        stage_data.append({
            'Stage': i + 1,
            'P_in (bara)': round(float(P_in), 1),
            'P_out (bara)': round(float(P_out), 1),
            'T_in (°C)': round(T_current, 1),
            'T_out (°C)': float('nan'), 'dT (°C)': float('nan'),
            'Phases': 0
        })

df = pd.DataFrame(stage_data)
print('Staged Depressurization Results:')
print(df.to_string(index=False))

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

stages = range(len(stage_data) + 1)
pressures = [P_initial] + [s['P_out (bara)'] for s in stage_data]
temps = [T_initial] + [s['T_out (°C)'] for s in stage_data]

ax1.step(stages, pressures, 'b-', linewidth=2, where='post')
ax1.plot(stages, pressures, 'bo', markersize=8)
ax1.set_ylabel('Pressure (bara)', fontsize=12)
ax1.set_title('Staged CO₂ Depressurization')
ax1.grid(alpha=0.3)

ax2.step(stages, temps, 'r-', linewidth=2, where='post')
ax2.plot(stages, temps, 'ro', markersize=8)
ax2.axhline(y=-46, color='gray', linestyle='--', alpha=0.7, label='MDMT (-46°C)')
ax2.axhline(y=-56.6, color='purple', linestyle='--', alpha=0.7, label='Triple point (-56.6°C)')
ax2.set_xlabel('Stage Number', fontsize=12)
ax2.set_ylabel('Temperature (°C)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('co2_staged_blowdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Effect of Impurities on Depressurization

Compare the depressurization behavior of pure CO₂ vs CO₂ with impurities.

In [ ]:
# Compare pure CO2 vs CO2 with impurities
compositions = [
    {'name': 'Pure CO₂', 'CO2': 1.0},
    {'name': '2% N₂', 'CO2': 0.98, 'nitrogen': 0.02},
    {'name': '5% N₂', 'CO2': 0.95, 'nitrogen': 0.05},
    {'name': '2% CH₄', 'CO2': 0.98, 'methane': 0.02},
    {'name': '2% H₂', 'CO2': 0.98, 'hydrogen': 0.02},
]

P_out_range = np.arange(6, 150, 2)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['black', 'blue', 'cyan', 'green', 'red']
linestyles = ['-', '-', '--', '-', '-']

for comp, color, ls in zip(compositions, colors, linestyles):
    temps = []
    valid_P = []

    for P_out in P_out_range:
        try:
            co2 = SystemSrkEos(273.15 + 40.0, 150.0)
            for k, v in comp.items():
                if k != 'name':
                    co2.addComponent(str(k), float(v))
            co2.setMixingRule('classic')

            f = Stream('F', co2)
            f.setFlowRate(100000.0, 'kg/hr')
            f.setTemperature(40.0, 'C')
            f.setPressure(150.0, 'bara')

            v = ThrottlingValve('V', f)
            v.setOutletPressure(float(P_out))

            p = ProcessSystem()
            p.add(f)
            p.add(v)
            p.run()

            T_out = float(v.getOutletStream().getTemperature()) - 273.15
            temps.append(T_out)
            valid_P.append(float(P_out))
        except Exception:
            pass

    ax.plot(valid_P, temps, linestyle=ls, color=color, linewidth=2, label=comp['name'])

ax.axhline(y=-56.6, color='purple', linestyle=':', alpha=0.7, label='Triple point')
ax.axhline(y=-46, color='gray', linestyle=':', alpha=0.7, label='MDMT (-46°C)')
ax.set_xlabel('Outlet Pressure (bara)', fontsize=12)
ax.set_ylabel('Outlet Temperature (°C)', fontsize=12)
ax.set_title('JT Expansion: Effect of Impurities (P₀=150 bar, T₀=40°C)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('co2_impurities_blowdown.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key takeaways for CO₂ depressurization design:

- **Dry ice risk**: Below 5.18 bara, solid CO₂ can form (triple point)
- **Minimum design temperature**: -46°C typical for carbon steel pipelines
- **Isentropic expansion** (rupture) produces colder conditions than JT (valve)
- **Impurities** (N₂, H₂) lower the cold temperature during expansion
- **Staged blowdown** limits minimum temperatures at each stage
- Design must ensure pipeline material can withstand the lowest possible temperature

### Design Recommendations
1. Never blowdown below 5.2 bara (above CO₂ triple point pressure)
2. Use controlled depressurization rate to allow heat transfer from surroundings
3. Consider 13% Cr or duplex stainless steel for cold sections
4. Account for impurities which shift the phase envelope

### References
- DNV-RP-J202: Design and Operation of CO₂ Pipelines
- API 521: Pressure-Relieving and Depressuring Systems
- Mahgerefteh et al. (2012): Modelling the impact of stream impurities on ductile fractures in CO₂ pipelines